# Baseline Interface

**Module:** Intelligent Interactive Systems

**Assignment:** Assignment 2, project report and prototype

**Name:** Leeya Hundal

**Student number:** 3183349

**Email:** LXH1038@student.bham.ac.uk

**Repository:** https://github.com/Leeya01/Assignment-A2-Project

**Submission release:** a2-submission

**Last updated:** 15/11/26


Local Region Image Editing is a prototype for Intelligent Interactive Systems (Assignment 2). The user paints over one area of an AI-generated image, types a short instruction, and only that area is edited, so a single wrong detail can be fixed without regenerating the whole picture. Everything outside the painted area is copied from the original, and a precision check confirms nothing else changed. The interface also shows before and after, warns about risky masks, offers three variants, keeps a history with undo, and records every edit in an exportable log.

In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
!pip install -q gradio

import gradio as gr
import numpy as np
from PIL import Image


import torch
from diffusers import AutoPipelineForInpainting

# use the GPU if there is one, otherwise fall back to the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Using device:", device)

# SDXL inpainting model, loaded in half precision to fit on the T4
sdxl_id = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
pipe_xl = AutoPipelineForInpainting.from_pretrained(
    sdxl_id, torch_dtype=dtype, variant="fp16"
)
pipe_xl = pipe_xl.to(device)
print("SDXL loaded")
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Using device: cuda


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

The config attributes {'decay': 0.9999, 'inv_gamma': 1.0, 'min_decay': 0.0, 'optimization_step': 37000, 'power': 0.6666666666666666, 'update_after_step': 0, 'use_ema_warmup': False} were passed to UNet2DConditionModel, but are not expected and will be ignored. Please verify your config.json configuration file.


SDXL loaded
/kaggle/input/datasets/leeyahundal/testimage2/Enchanted Underwater Pirate Shipwreck.png


Section 2: Brush. The user paints over the flaw and the app turns the strokes into a black and white mask (white is the area to fix). I show the mask as a preview to check it matches what was painted.

In [5]:


def get_mask(editor_value):
    # painted strokes sit on the first layer, so I use its alpha channel as the mask
    if editor_value is None or len(editor_value["layers"]) == 0:
        return None
    layer = np.array(editor_value["layers"][0])
    mask = (layer[:, :, 3] > 0).astype(np.uint8) * 255
    return Image.fromarray(mask)

with gr.Blocks() as demo:
    gr.Markdown("Inpainting editor")
    editor = gr.ImageEditor(
        type="pil",
        label="Upload an image and paint over the flaw",
        brush=gr.Brush(colors=["#ffffff"], default_size=20),
    )
    mask_out = gr.Image(label="Mask preview")
    show_btn = gr.Button("Show mask")
    show_btn.click(get_mask, editor, mask_out)

demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b8ff5bf94c0ba58c67.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b8ff5bf94c0ba58c67.gradio.live


Section 3: Instruction box and apply button. The user paints the flaw, types what should go there, and presses apply. The app builds the mask from the brush strokes, runs SDXL inpainting on that area only, then pastes the result back with a softened edge so everything outside the mask stays identical. It also prints the average change outside the mask as the precision check.

In [6]:
import gradio as gr
import numpy as np
import torch
from PIL import Image, ImageFilter

def get_mask(editor_value):
    # painted strokes sit on the first layer, so I use its alpha channel as the mask
    if editor_value is None or len(editor_value["layers"]) == 0:
        return None
    layer = np.array(editor_value["layers"][0])
    mask = (layer[:, :, 3] > 0).astype(np.uint8) * 255
    return Image.fromarray(mask).resize((512, 512), Image.NEAREST)

def apply_edit(editor_value, prompt):
    if editor_value is None or editor_value["background"] is None:
        raise gr.Error("Upload an image first.")
    if not prompt or not prompt.strip():
        raise gr.Error("Type what should appear in the painted area.")
    mask = get_mask(editor_value)
    if mask is None or np.array(mask).max() == 0:
        raise gr.Error("Paint over the area you want to change first.")

    base = editor_value["background"].convert("RGB").resize((512, 512))
    soft = mask.filter(ImageFilter.GaussianBlur(5))

    # SDXL works at 1024, so scale up, edit, then scale back down
    out = pipe_xl(
        prompt=prompt,
        negative_prompt="blurry, text, watermark, realistic photo, harsh shadows",
        image=base.resize((1024, 1024)),
        mask_image=mask.resize((1024, 1024)),
        padding_mask_crop=64,
        guidance_scale=5.5,
        num_inference_steps=30,
        generator=torch.Generator(device).manual_seed(42),
    ).images[0].resize((512, 512))

    # paste back so everything outside the mask matches the original
    final = Image.composite(out, base, soft)

    # precision check, average change outside the mask on a 0 to 255 scale
    outside = np.array(soft) == 0
    diff = np.abs(np.array(final).astype(float) - np.array(base).astype(float)).mean(axis=2)
    note = "Average change outside the mask: " + str(round(diff[outside].mean(), 3))
    return final, note

with gr.Blocks() as demo:
    gr.Markdown("Inpainting editor")
    editor = gr.ImageEditor(
        type="pil",
        label="Upload an image and paint over the area to change",
        brush=gr.Brush(colors=["#ffffff"], default_size=20),
    )
    prompt_box = gr.Textbox(label="What should go in the painted area?")
    apply_btn = gr.Button("Apply edit")
    result = gr.Image(label="Result")
    check = gr.Textbox(label="Precision check")
    apply_btn.click(apply_edit, [editor, prompt_box], [result, check])

demo.launch(share=True, debug=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://9e09d823dae0cd98af.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

... (15 outputs left out)

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://9e09d823dae0cd98af.gradio.live


In [ ]:
def use_result(img):
    if img is None:
        raise gr.Error("Apply an edit first.")
    # returning an image puts it in the editor as the new base image
    return img

with gr.Blocks() as demo:
    gr.Markdown("Inpainting editor")
    editor = gr.ImageEditor(
        type="pil",
        label="Upload an image and paint over the area to change",
        brush=gr.Brush(colors=["#ffffff"], default_size=20),
    )
    prompt_box = gr.Textbox(label="What should go in the painted area?")
    apply_btn = gr.Button("Apply edit")
    result = gr.Image(label="Result")
    check = gr.Textbox(label="Precision check")
    again_btn = gr.Button("Keep editing this result")
    apply_btn.click(apply_edit, [editor, prompt_box], [result, check])
    again_btn.click(use_result, result, editor)

demo.launch(share=True, debug=True)